# Fine-Tuning: Parameter Selection via Grid Search

Systematic grid search for **Modified Newton** and **Truncated Newton** method parameters.

**Two-phase approach:**
1. **Phase 1** â€” Tune method parameters (Armijo backtracking, Cholesky modification, forcing terms) with a fixed stopping criterion (`GradNormAbsolute(1e-8)`). The stopping criterion does not affect the convergence trajectory â€” it only decides when to stop.
2. **Phase 2** â€” Analyze stopping criteria using 3 tolerance bands (rough / good / very good) adapted per criterion type.

Uses **exact derivatives only** (finite difference variants are studied separately in Assignment Section 3).

---

**Table of Contents**
1. [Setup & Configuration](#setup)
2. [Helpers & Infrastructure](#helpers)
3. [Phase 1: Modified Newton Grid Search](#phase1-mn)
4. [Phase 1: Truncated Newton Grid Search](#phase1-tn)
5. [Phase 1: Results & Visualization](#phase1-results)
6. [Phase 2: Stopping Criteria Analysis](#phase2)
7. [Export & Summary](#export)

<a id="setup"></a>
## 1. Setup & Configuration

In [1]:
import sys
import time
import itertools
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.functions import f16, f28, x_bar_16, x_bar_28
from src.gradients import grad_f16, grad_f28
from src.hessians import hess_f16, hess_f28
from src.methods.modified_newton import modified_newton
from src.methods.truncated_newton import truncated_newton
from src.starting_points import generate_starting_points
from src.stopping_criteria import (
    StoppingCriterion,
    GradNormAbsolute, GradNormRelative,
    FChangeAbsolute, FChangeRelative,
    XChangeAbsolute, XChangeRelative,
)

In [2]:
# CONFIGURATION
SEED = min(346165, 323334)

TIME_LIMIT = 20

QUICK_MODE = False   # Set False for full experiment (1-4 hours)

if QUICK_MODE:
    DIMENSIONS     = [2, 1000]
    NUM_RANDOM     = 1          # x_bar + 1 random = 2 starting points
    MAX_ITER       = 200
else:
    DIMENSIONS     = [2, 1000, 10_000, 100_000]
    NUM_RANDOM     = 5          # x_bar + 5 random = 6 starting points
    MAX_ITER       = 1000

OOM_THRESHOLD_MB = 4096         # skip cells where dense Hessian exceeds 4 GB

PROBLEMS = {
    'P16': dict(f=f16, grad=grad_f16, hess=hess_f16,
                x_bar=x_bar_16, desc='Banded Trigonometric (diagonal H)'),
    'P28': dict(f=f28, grad=grad_f28, hess=hess_f28,
                x_bar=x_bar_28, desc='Variably Dimensioned (dense H)'),
}

print(f"SEED           = {SEED}")
print(f"QUICK_MODE     = {QUICK_MODE}")
print(f"DIMENSIONS     = {DIMENSIONS}")
print(f"Starting pts   = {1 + NUM_RANDOM} per (problem, n)")
print(f"MAX_ITER       = {MAX_ITER}")
print(f"OOM threshold  = {OOM_THRESHOLD_MB} MB")


SEED           = 323334
QUICK_MODE     = False
DIMENSIONS     = [2, 1000, 10000, 100000]
Starting pts   = 6 per (problem, n)
MAX_ITER       = 1000
OOM threshold  = 4096 MB


<a id="helpers"></a>
## 2. Helpers & Infrastructure

In [3]:
# Combined Stopping Criterion (OR logic)

class CombinedStoppingCriterion(StoppingCriterion):
    """OR-combination: fires when ANY inner criterion fires."""

    def __init__(self, criteria: list):
        self.criteria = criteria
        self.tol = None
        self._triggered = "max_iter"

    @property
    def name(self):
        return self._triggered

    def initialize(self, x0, F0, g0):
        for c in self.criteria:
            c.initialize(x0, F0, g0)

    def should_stop(self, k, x, F, g, x_prev, F_prev) -> bool:
        for c in self.criteria:
            if c.should_stop(k, x, F, g, x_prev, F_prev):
                self._triggered = c.name
                return True
        return False

    def __repr__(self):
        return f"Combined({self.criteria})"

class CombinedRelStoppingCriterion:
    """OR-combination of relative criteria. Mirrors CombinedStoppingCriterion."""
    def __init__(self, grad_crit=None, f_crit=None, x_crit=None):
        self.criteria = [c for c in (grad_crit, f_crit, x_crit) if c is not None]
        self._triggered = None
        self.name = 'combined_rel'

    def should_stop(self, k, x, F, g, x_prev=None, F_prev=None):
        for c in self.criteria:
            if c.should_stop(k, x, F, g, x_prev=x_prev, F_prev=F_prev):
                self._triggered = getattr(c, 'name', type(c).__name__)
                return True
        return False


In [4]:
# OOM guard

def expected_hessian_mb(n):
    """Memory for a dense (n, n) float64 matrix in MB."""
    return n * n * 8 / (1024 ** 2)

def should_skip(prob_id, n, method='any'):
    """Skip if the combination would OOM or be infeasible.

    P16 has a sparse diagonal Hessian -> never OOM.
    P28 has a dense Hessian -> skip Modified Newton at large n.
    Truncated Newton on P28 can use matrix-free Hv products.
    """
    if prob_id == 'P16':
        return False
    dense_mb = expected_hessian_mb(n)
    if method == 'truncated_newton':
        return False
    return dense_mb > OOM_THRESHOLD_MB

# Preview
print("OOM preview:")
for pid in ('P16', 'P28'):
    for n in DIMENSIONS:
        mb = expected_hessian_mb(n)
        mn_skip = should_skip(pid, n, 'modified_newton')
        tn_skip = should_skip(pid, n, 'truncated_newton')
        mn_tag = 'SKIP' if mn_skip else 'OK'
        tn_tag = 'SKIP' if tn_skip else 'OK'
        print(f"  {pid} n={n:>6d}: {mb:>10.0f} MB  MN={mn_tag:>4s}  TN={tn_tag:>4s}")



OOM preview:
  P16 n=     2:          0 MB  MN=  OK  TN=  OK
  P16 n=  1000:          8 MB  MN=  OK  TN=  OK
  P16 n= 10000:        763 MB  MN=  OK  TN=  OK
  P16 n=100000:      76294 MB  MN=  OK  TN=  OK
  P28 n=     2:          0 MB  MN=  OK  TN=  OK
  P28 n=  1000:          8 MB  MN=  OK  TN=  OK
  P28 n= 10000:        763 MB  MN=  OK  TN=  OK
  P28 n=100000:      76294 MB  MN=SKIP  TN=  OK


In [5]:
# Experimental convergence rate estimator
# Estimates the convergence order p from the ||g_k|| sequence via
# successive log-ratios: p_k = log||g_{k+1}|| / log||g_k||.
# p~1 -> linear, p~1.5 -> superlinear, p~2 -> quadratic.

def experimental_rate(g_norms):
    """Estimate convergence order p from ||g_k|| sequence.

    Model: ||g_{k+1}|| ~ C * ||g_k||^p
    Returns median of tail estimates. NaN if not computable.
    """
    e = np.asarray(g_norms, dtype=float)
    if e.size < 4:
        return float('nan')
    e = e[e > 1e-14]
    if e.size < 4:
        return float('nan')
    log_e = np.log(e)
    num = np.diff(log_e)[1:]
    den = np.diff(log_e)[:-1]
    mask = (np.abs(den) > 1e-12) & np.isfinite(num) & np.isfinite(den)
    if not mask.any():
        return float('nan')
    p_vals = num[mask] / den[mask]
    p_tail = p_vals[-min(5, p_vals.size):]
    return float(np.median(p_tail))


# Aggregation helper
#
# Two layers:
#   1. agg     -> per (params, problem, n): success_rate (all runs) +
#                 iter/grad/time means computed ONLY on successful runs.
#                 This avoids the failed-run grad_norm (huge values like 1e27
#                 because the algorithm hit max_iter without converging) from
#                 polluting the means.
#   2. summary -> per (params): kept backward-compatible columns
#                 (avg_success, avg_iter, avg_time, avg_grad) PLUS new
#                 split-by-problem columns (success_P16/P28, iter_P16/P28,
#                 time_P16/P28). The split columns are what actually justify
#                 the choice of best parameters.

def aggregate_grid(df, param_cols):
    """Aggregate grid search results, split by problem."""
    group_inner = param_cols + ['problem', 'n']
    has_cg = 'cg_total' in df.columns
    ok = df[df['success']]
    sr = df.groupby(group_inner).agg(
        success_rate=('success', 'mean'),
        n_runs=('success', 'count'),
    ).reset_index()
    q_aggs = dict(
        mean_iter=('n_iter', 'mean'),
        std_iter=('n_iter', 'std'),
        mean_grad=('grad_norm', 'mean'),
        mean_time=('time_s', 'mean'),
        mean_backtrack=('n_backtrack', 'mean'),
    )
    if has_cg:
        q_aggs['mean_cg'] = ('cg_total', 'mean')
    q = ok.groupby(group_inner).agg(**q_aggs).reset_index()
    agg = sr.merge(q, on=group_inner, how='left')

    # Per-problem split
    parts = []
    for prob in sorted(agg['problem'].unique()):
        sub = agg[agg['problem'] == prob]
        split_aggs = {
            f'success_{prob}':   ('success_rate', 'mean'),
            f'iter_{prob}':      ('mean_iter', 'mean'),
            f'time_{prob}':      ('mean_time', 'mean'),
            f'backtrack_{prob}': ('mean_backtrack', 'mean'),
        }
        if has_cg:
            split_aggs[f'cg_{prob}'] = ('mean_cg', 'mean')
        p = sub.groupby(param_cols).agg(**split_aggs).reset_index()
        parts.append(p)
    summary = parts[0]
    for p in parts[1:]:
        summary = summary.merge(p, on=param_cols, how='outer')

    # Backward-compat aggregated columns (filtered semantics now)
    overall_all = df.groupby(param_cols).agg(
        avg_success=('success', 'mean'),
    ).reset_index()
    overall_aggs = dict(
        avg_iter=('n_iter', 'mean'),
        avg_time=('time_s', 'mean'),
        avg_grad=('grad_norm', 'mean'),
        avg_backtrack=('n_backtrack', 'mean'),
    )
    if has_cg:
        overall_aggs['avg_cg'] = ('cg_total', 'mean')
    overall_ok = ok.groupby(param_cols).agg(**overall_aggs).reset_index()
    summary = summary.merge(overall_all, on=param_cols, how='left')
    summary = summary.merge(overall_ok, on=param_cols, how='left')

    # Selection hierarchy (set by user, see docs/tuning_analysis.md §6):
    #   1) avg_success  (does the method converge?)
    #   2) avg_iter     (efficiency, success-only)
    #   3) avg_time     (wall-clock tiebreaker)
    # avg_grad is NOT used for ranking: on successful runs it is bounded by
    # the fixed tol of the Phase-1 stop criterion (1e-4), so it does not
    # discriminate between configurations. It is kept as a sanity check.
    summary = summary.sort_values(
        by=['avg_success', 'avg_iter', 'avg_time'],
        ascending=[False, True, True]
    ).reset_index(drop=True)
    summary.index.name = 'rank'
    return summary, agg


def pivot_success_by_dim(agg, param_cols):
    """success_rate per (problem, n) x (params)."""
    return agg.pivot_table(index=['problem', 'n'], columns=param_cols,
                           values='success_rate')


def pivot_iter_by_dim(agg, param_cols):
    """mean_iter (success-only) per (problem, n) x (params)."""
    return agg.pivot_table(index=['problem', 'n'], columns=param_cols,
                           values='mean_iter')


def best_for_problem(df, param_cols, problem):
    """Best params filtered to a single problem.

    Calls aggregate_grid on the df subset for one problem and returns the
    top row of the ranking (sorted by avg_success, avg_iter, avg_time).
    """
    sub = df[df['problem'] == problem]
    s, _ = aggregate_grid(sub, param_cols)
    return s.iloc[0]


def best_overall(summary):
    """Top of the global ranking (summary already sorted)."""
    return summary.iloc[0]


In [6]:
# alpha_min analysis
# For each (rho, max_iter_backtrack) pair, the smallest achievable step is
# alpha_min = rho^T * alpha_0.  With alpha_0 = 1 this is just rho^T.
# Classification: 'safe' means alpha_min < eps_mach (~1e-15),
# i.e. backtracking can effectively reach zero; 'limited' otherwise.

rho_vals_preview = [0.3, 0.5, 0.8]
T_vals_preview = [30, 50, 100, 150, 200]

rows_alpha = []
for rho in rho_vals_preview:
    for T in T_vals_preview:
        a_min = rho ** T
        rows_alpha.append(dict(rho=rho, T=T, alpha_min=a_min,
                               log10_alpha=np.log10(a_min) if a_min > 0 else -np.inf))

alpha_df = pd.DataFrame(rows_alpha)

alpha_df['zone'] = alpha_df['alpha_min'].apply(
    lambda a: 'safe (< eps_mach)' if a < 1e-15 else 'limited')

print("Minimum achievable step: alpha_min = rho^T")
print(alpha_df.to_string(index=False))

Minimum achievable step: alpha_min = rho^T
 rho   T     alpha_min  log10_alpha              zone
 0.3  30  2.058911e-16   -15.686362 safe (< eps_mach)
 0.3  50  7.178980e-27   -26.143937 safe (< eps_mach)
 0.3 100  5.153775e-53   -52.287875 safe (< eps_mach)
 0.3 150  3.699885e-79   -78.431812 safe (< eps_mach)
 0.3 200 2.656140e-105  -104.575749 safe (< eps_mach)
 0.5  30  9.313226e-10    -9.030900           limited
 0.5  50  8.881784e-16   -15.051500 safe (< eps_mach)
 0.5 100  7.888609e-31   -30.103000 safe (< eps_mach)
 0.5 150  7.006492e-46   -45.154499 safe (< eps_mach)
 0.5 200  6.223015e-61   -60.205999 safe (< eps_mach)
 0.8  30  1.237940e-03    -2.907300           limited
 0.8  50  1.427248e-05    -4.845501           limited
 0.8 100  2.037036e-10    -9.691001           limited
 0.8 150  2.907355e-15   -14.536502           limited
 0.8 200  4.149516e-20   -19.382003 safe (< eps_mach)


In [7]:
# Starting points generation

starts_cache = {}
rng = np.random.default_rng(SEED)

for prob_id in ('P16', 'P28'):
    x_bar_fn = PROBLEMS[prob_id]['x_bar']
    for n in DIMENSIONS:
        pts = generate_starting_points(x_bar_fn(n), num_random=NUM_RANDOM, rng=rng)
        starts_cache[(prob_id, n)] = pts
        print(f"  {prob_id} n={n:>6d}: {len(pts)} points "
              f"(||x_bar||={np.linalg.norm(pts[0]):.4f})")

print(f"\nTotal: {sum(len(v) for v in starts_cache.values())} starting points cached")

  P16 n=     2: 6 points (||x_bar||=1.4142)
  P16 n=  1000: 6 points (||x_bar||=31.6228)
  P16 n= 10000: 6 points (||x_bar||=100.0000)
  P16 n=100000: 6 points (||x_bar||=316.2278)
  P28 n=     2: 6 points (||x_bar||=0.5000)
  P28 n=  1000: 6 points (||x_bar||=18.2437)
  P28 n= 10000: 6 points (||x_bar||=57.7307)
  P28 n=100000: 6 points (||x_bar||=182.5728)

Total: 48 starting points cached


<a id="phase1-mn"></a>
## 3. Phase 1: Modified Newton — Grid Search

**Fixed stopping criterion:** `GradNormAbsolute(1e-8)` ("good solution" band).

**Parameters tuned** (see `docs/tuning_analysis.md` for theoretical justification):
| Parameter | Values | Description |
|-----------|--------|-------------|
| `beta` | 1e-6, 1e-3 | Cholesky modification heuristic |
| `rho` | 0.5, 0.8 | Backtracking reduction factor |

**Fixed parameters** (theoretically insensitive — see analysis):
| Parameter | Value | Reason |
|-----------|-------|--------|
| `alpha0` | 1 | Newton step, prerequisite for quadratic convergence |
| `c1` | 1e-4 | Nocedal-Wright standard; 5000:1 margin on full Newton step |
| `max_tau_iter` | 100 | Safety; ≤25 doublings ever needed |
| `max_iter_backtrack` | 50 | Safety; 50 steps → α ≈ 1e-15 |

In [8]:
# Modified Newton parameter grid (reduced — see docs/tuning_analysis.md)

MN_GRID = dict(
    beta = [1e-6, 1e-3, 1e-2],
    rho  = [0.5, .75, 0.9],
)

# Fixed Armijo & safety parameters
MN_FIXED = dict(c1=1e-4, max_tau_iter=100, max_iter_backtrack=50)

mn_keys = list(MN_GRID.keys())
mn_combos = list(itertools.product(*MN_GRID.values()))

n_valid_cells = sum(
    len(starts_cache[(pid, n)])
    for pid in ('P16', 'P28') for n in DIMENSIONS
    if not should_skip(pid, n, 'modified_newton')
)

print(f"Grid: {' x '.join(str(len(v)) for v in MN_GRID.values())} "
      f"= {len(mn_combos)} combos")
print(f"Valid (problem, n, start) cells: {n_valid_cells}")
print(f"Total runs: {len(mn_combos) * n_valid_cells}")

Grid: 3 x 3 = 9 combos
Valid (problem, n, start) cells: 42
Total runs: 378


In [ ]:
# Modified Newton grid search execution

mn_rows = []
total = len(mn_combos) * n_valid_cells
idx = 0
t0_total = time.perf_counter()
print(f"Modified Newton: {total} experiments ({len(mn_combos)} combos x {n_valid_cells} cells)")
print("-" * 80)

for combo in mn_combos:
    beta, rho = combo

    for prob_id in ('P16', 'P28'):
        pinfo = PROBLEMS[prob_id]
        for n in DIMENSIONS:
            if should_skip(prob_id, n, 'modified_newton'):
                continue
            for si, x0 in enumerate(starts_cache[(prob_id, n)]):
                idx += 1
                stop = GradNormAbsolute(tol=1e-8)
                t0 = time.perf_counter()
                try:
                    res = modified_newton(
                        pinfo['f'], x0, stop,
                        grad_f=pinfo['grad'], hess_f=pinfo['hess'],
                        alpha0=1.0, c1=MN_FIXED['c1'], rho=rho,
                        beta=beta, max_tau_iter=MN_FIXED['max_tau_iter'],
                        max_iter=MAX_ITER,
                        max_iter_backtrack=MN_FIXED['max_iter_backtrack'],
                        return_history=False, time_limit=TIME_LIMIT)
                    elapsed = time.perf_counter() - t0
                    tag = 'OK' if res['success'] else 'FAIL'
                    print(f"[{idx:>4d}/{total}] {tag}  {prob_id} n={n:<6d} s={si} "
                          f"rho={rho} beta={beta:.0e} | "
                          f"it={res['n_iter']:>3d} ||g||={res['grad_norm']:.2e} "
                          f"t={elapsed:.2f}s")
                    row = dict(
                        problem=prob_id, n=n, start_idx=si,
                        beta=beta, rho=rho,
                        n_iter=res['n_iter'], success=res['success'],
                        grad_norm=res['grad_norm'], f_star=res['f_star'],
                        time_s=elapsed,
                        stop_reason=res['stop_reason'],
                        chol_adj=res.get('n_chol_adjustments_total', 0), n_backtrack=res.get('n_backtrack_total', 0))
                except MemoryError:
                    print(f"[{idx:>4d}/{total}] OOM  {prob_id} n={n:<6d} s={si} "
                          f"rho={rho} beta={beta:.0e}")
                    row = dict(
                        problem=prob_id, n=n, start_idx=si,
                        beta=beta, rho=rho,
                        n_iter=0, success=False,
                        grad_norm=np.nan, f_star=np.nan,
                        time_s=time.perf_counter() - t0,
                        stop_reason='memory_error', chol_adj=0, n_backtrack=0)
                except Exception as e:
                    print(f"[{idx:>4d}/{total}] ERR  {prob_id} n={n:<6d} s={si} "
                          f"rho={rho} beta={beta:.0e} | {e}")
                    row = dict(
                        problem=prob_id, n=n, start_idx=si,
                        beta=beta, rho=rho,
                        n_iter=0, success=False,
                        grad_norm=np.nan, f_star=np.nan,
                        time_s=time.perf_counter() - t0,
                        stop_reason=f'err:{type(e).__name__}', chol_adj=0, n_backtrack=0)
                mn_rows.append(row)

mn_df = pd.DataFrame(mn_rows)
elapsed_total = time.perf_counter() - t0_total
print("-" * 80)
print(f"Modified Newton DONE: {len(mn_df)} runs in {elapsed_total:.1f}s, "
      f"{mn_df['success'].sum()} successes "
      f"({mn_df['success'].mean()*100:.1f}%)")

Modified Newton: 378 experiments (9 combos x 42 cells)
--------------------------------------------------------------------------------
[   1/378] OK  P16 n=2      s=0 rho=0.5 beta=1e-06 | it=  5 ||g||=4.18e-13 t=0.00s
[   2/378] OK  P16 n=2      s=1 rho=0.5 beta=1e-06 | it=  4 ||g||=4.53e-13 t=0.00s
[   3/378] OK  P16 n=2      s=2 rho=0.5 beta=1e-06 | it=  4 ||g||=4.03e-12 t=0.00s
[   4/378] OK  P16 n=2      s=3 rho=0.5 beta=1e-06 | it=  6 ||g||=7.82e-12 t=0.00s
[   5/378] OK  P16 n=2      s=4 rho=0.5 beta=1e-06 | it=  6 ||g||=7.89e-14 t=0.00s
[   6/378] OK  P16 n=2      s=5 rho=0.5 beta=1e-06 | it=  3 ||g||=4.29e-11 t=0.00s
[   7/378] OK  P16 n=1000   s=0 rho=0.5 beta=1e-06 | it=  6 ||g||=4.35e-10 t=0.00s
[   8/378] FAIL  P16 n=1000   s=1 rho=0.5 beta=1e-06 | it=1000 ||g||=3.28e-08 t=0.29s
[   9/378] FAIL  P16 n=1000   s=2 rho=0.5 beta=1e-06 | it=1000 ||g||=1.28e-07 t=0.32s
[  10/378] FAIL  P16 n=1000   s=3 rho=0.5 beta=1e-06 | it=1000 ||g||=1.88e-07 t=0.29s
[  11/378] FAIL  P16 n=10

In [ ]:
# Modified Newton: aggregation and ranking

mn_param_cols = ['beta', 'rho']
mn_summary, mn_detail = aggregate_grid(mn_df, mn_param_cols)

print("=== Modified Newton: Ranking ===")
print("Sort: avg_success DESC -> avg_iter ASC -> avg_time ASC")
print("(avg_grad column is a sanity check; not used for sorting)")
display(mn_summary)

# Best parameters: 3 selections (global, P16-optimized, P28-optimized).
best_mn = best_overall(mn_summary)
best_mn_P16 = best_for_problem(mn_df, mn_param_cols, 'P16')
best_mn_P28 = best_for_problem(mn_df, mn_param_cols, 'P28')

print("\n=== Best parameters ===")
print(f"  Overall  : beta={best_mn['beta']:.0e}, rho={best_mn['rho']}  "
      f"(success={best_mn['avg_success']:.2%}, "
      f"iter={best_mn['avg_iter']:.1f}, time={best_mn['avg_time']:.2f}s)")
print(f"  Best P16 : beta={best_mn_P16['beta']:.0e}, rho={best_mn_P16['rho']}  "
      f"(success_P16={best_mn_P16['success_P16']:.2%}, "
      f"iter_P16={best_mn_P16['iter_P16']:.1f})")
print(f"  Best P28 : beta={best_mn_P28['beta']:.0e}, rho={best_mn_P28['rho']}  "
      f"(success_P28={best_mn_P28['success_P28']:.2%}, "
      f"iter_P28={best_mn_P28['iter_P28']:.1f})")

# Pivot tables that justify the choices.
print("\n=== Success rate per (problem, n) x (beta, rho) ===")
display(pivot_success_by_dim(mn_detail, mn_param_cols))

print("\n=== Mean iterations (success-only) per (problem, n) x (beta, rho) ===")
display(pivot_iter_by_dim(mn_detail, mn_param_cols))

print("\nNote: NaN in 'mean_iter' = (problem, n) cells where ALL runs failed")
print("and were excluded by the success-filter. Inspect the success_rate")
print("pivot to see WHERE the algorithm fails. Typical causes on this grid:")
print("  - P16 @ n=100000     : max_iter (1000 iter not enough)")
print("  - P28 @ n>=10000     : time_limit (60s not enough; algorithm still")
print("                          converging, just slowly)")


=== Modified Newton: Ranking ===
Sort: avg_success DESC -> avg_iter ASC -> avg_time ASC
(avg_grad column is a sanity check; not used for sorting)


,beta,rho,success_P16,iter_P16,time_P16,success_P28,iter_P28,time_P28,avg_success,avg_iter,avg_time,avg_grad
rank,,,,,,,,,,,,
0,0.000001,0.8,0.416667,12.666667,0.030556,0.666667,21.5,3.446598,0.52381,16.818182,1.888848,4.494035e-10
1,0.000001,0.5,0.375000,5.666667,0.024045,0.666667,21.5,3.355545,0.50000,14.476190,1.922475,9.773671e-11
2,0.001000,0.5,0.375000,6.458333,0.016678,0.666667,21.5,2.476366,0.50000,14.666667,1.418605,3.325988e-11
3,0.001000,0.8,0.375000,6.458333,0.017103,0.666667,21.5,2.533897,0.50000,14.666667,1.451561,3.326176e-11



=== Best parameters ===
  Overall  : beta=1e-06, rho=0.8  (success=52.38%, iter=16.8, time=1.89s)
  Best P16 : beta=1e-06, rho=0.8  (success_P16=41.67%, iter_P16=12.7)
  Best P28 : beta=1e-03, rho=0.5  (success_P28=66.67%, iter_P28=21.5)

=== Success rate per (problem, n) x (beta, rho) ===


beta            0.000001            0.001000          
rho                  0.5       0.8       0.5       0.8
problem n                                             
P16     2       1.000000  1.000000  1.000000  1.000000
        1000    0.166667  0.333333  0.166667  0.166667
        10000   0.166667  0.166667  0.166667  0.166667
        100000  0.166667  0.166667  0.166667  0.166667
P28     2       1.000000  1.000000  1.000000  1.000000
        1000    1.000000  1.000000  1.000000  1.000000
        10000   0.000000  0.000000  0.000000  0.000000


=== Mean iterations (success-only) per (problem, n) x (beta, rho) ===


beta             0.000001              0.001000           
rho                   0.5        0.8        0.5        0.8
problem n                                                 
P16     2        4.666667   5.666667   4.833333   4.833333
        1000     6.000000  33.000000   7.000000   7.000000
        10000    6.000000   6.000000   7.000000   7.000000
        100000   6.000000   6.000000   7.000000   7.000000
P28     2        6.000000   6.000000   6.000000   6.000000
        1000    37.000000  37.000000  37.000000  37.000000


Note: NaN in 'mean_iter' = (problem, n) cells where ALL runs failed
and were excluded by the success-filter. Inspect the success_rate
pivot to see WHERE the algorithm fails. Typical causes on this grid:
  - P16 @ n=100000     : max_iter (1000 iter not enough)
  - P28 @ n>=10000     : time_limit (60s not enough; algorithm still
                          converging, just slowly)


<a id="phase1-tn"></a>
## 4. Phase 1: Truncated Newton — Grid Search

**Fixed stopping criterion:** `GradNormAbsolute(1e-8)`.

**Parameters tuned** (see `docs/tuning_analysis.md`):
| Parameter | Values | Description |
|-----------|--------|-------------|
| `forcing` | superlinear, quadratic | Forcing sequence for inner CG tolerance $\eta_k$ |
| `rho` | 0.5, 0.8 | Backtracking reduction factor |

**Fixed parameters:**
| Parameter | Value | Reason |
|-----------|-------|--------|
| `alpha0` | 1 | Newton step |
| `c1` | 1e-4 | Standard; insensitive |
| `cg_max_iter` | None (=n) | CG converges in 1-2 iter for P16/P28 |
| `max_iter_backtrack` | 50 | Safety |

**Forcing sequences** (Theorem 6.2 in [SW]):
- `superlinear`: $\eta_k = \min(0.5, \sqrt{\|\nabla f(x_k)\|})$ → superlinear rate
- `quadratic`: $\eta_k = \min(0.5, \|\nabla f(x_k)\|)$ → quadratic rate
- `linear` excluded: constant $\eta=0.5$ → linear convergence, never competitive

In [ ]:
# Truncated Newton parameter grid (reduced — see docs/tuning_analysis.md)

TN_GRID = dict(
    forcing = ['superlinear', 'quadratic'],
    rho     = [.5, .75, .9],
)

# Fixed Armijo & CG parameters
TN_FIXED = dict(c1=1e-4, cg_max_iter=None, max_iter_backtrack=50)

tn_keys = list(TN_GRID.keys())
tn_combos = list(itertools.product(*TN_GRID.values()))

tn_valid_cells = sum(
    len(starts_cache[(pid, n)])
    for pid in ('P16', 'P28') for n in DIMENSIONS
    if not should_skip(pid, n, 'truncated_newton')
)

print(f"Grid: {' x '.join(str(len(v)) for v in TN_GRID.values())} "
      f"= {len(tn_combos)} combos")
print(f"Valid (problem, n, start) cells: {tn_valid_cells}")
print(f"Total runs: {len(tn_combos) * tn_valid_cells}")

Grid: 2 x 3 = 6 combos
Valid (problem, n, start) cells: 48
Total runs: 288


In [ ]:
# Truncated Newton grid search execution

tn_rows = []
total_tn = len(tn_combos) * tn_valid_cells
idx = 0
t0_total = time.perf_counter()
print(f"Truncated Newton: {total_tn} experiments ({len(tn_combos)} combos x {tn_valid_cells} cells)")
print("-" * 80)

for combo in tn_combos:
    forcing, rho = combo

    for prob_id in ('P16', 'P28'):
        pinfo = PROBLEMS[prob_id]
        for n in DIMENSIONS:
            if should_skip(prob_id, n, 'truncated_newton'):
                continue
            for si, x0 in enumerate(starts_cache[(prob_id, n)]):
                idx += 1
                stop = GradNormAbsolute(tol=1e-8)
                t0 = time.perf_counter()
                try:
                    res = truncated_newton(
                        pinfo['f'], x0, stop,
                        grad_f=pinfo['grad'],
                        hess_f=(pinfo['hess'] if expected_hessian_mb(n) <= OOM_THRESHOLD_MB else None),
                        alpha0=1.0, c1=TN_FIXED['c1'], rho=rho,
                        forcing=forcing, cg_max_iter=TN_FIXED['cg_max_iter'],
                        max_iter=MAX_ITER,
                        max_iter_backtrack=TN_FIXED['max_iter_backtrack'],
                        return_history=False, time_limit=TIME_LIMIT)
                    elapsed = time.perf_counter() - t0
                    tag = 'OK' if res['success'] else 'FAIL'
                    print(f"[{idx:>4d}/{total_tn}] {tag}  {prob_id} n={n:<6d} s={si} "
                          f"rho={rho} f={forcing[:5]} | "
                          f"it={res['n_iter']:>3d} ||g||={res['grad_norm']:.2e} "
                          f"cg_tot={res.get('cg_iters_total',0)} "
                          f"t={elapsed:.2f}s")
                    row = dict(
                        problem=prob_id, n=n, start_idx=si,
                        forcing=forcing, rho=rho,
                        n_iter=res['n_iter'], success=res['success'],
                        grad_norm=res['grad_norm'], f_star=res['f_star'],
                        time_s=elapsed,
                        stop_reason=res['stop_reason'],
                        cg_total=res.get('cg_iters_total', 0),
                        neg_curv=res.get('neg_curvature_count', 0), n_backtrack=res.get('n_backtrack_total', 0))
                except MemoryError:
                    print(f"[{idx:>4d}/{total_tn}] OOM  {prob_id} n={n:<6d} s={si}")
                    row = dict(
                        problem=prob_id, n=n, start_idx=si,
                        forcing=forcing, rho=rho,
                        n_iter=0, success=False,
                        grad_norm=np.nan, f_star=np.nan,
                        time_s=time.perf_counter() - t0,
                        stop_reason='memory_error',
                        cg_total=0, neg_curv=0, n_backtrack=0)
                except Exception as e:
                    print(f"[{idx:>4d}/{total_tn}] ERR  {prob_id} n={n:<6d} s={si} | {e}")
                    row = dict(
                        problem=prob_id, n=n, start_idx=si,
                        forcing=forcing, rho=rho,
                        n_iter=0, success=False,
                        grad_norm=np.nan, f_star=np.nan,
                        time_s=time.perf_counter() - t0,
                        stop_reason=f'err:{type(e).__name__}',
                        cg_total=0, neg_curv=0, n_backtrack=0)
                tn_rows.append(row)

tn_df = pd.DataFrame(tn_rows)
elapsed_total = time.perf_counter() - t0_total
print("-" * 80)
print(f"Truncated Newton DONE: {len(tn_df)} runs in {elapsed_total:.1f}s, "
      f"{tn_df['success'].sum()} successes "
      f"({tn_df['success'].mean()*100:.1f}%)")

Truncated Newton: 288 experiments (6 combos x 48 cells)
--------------------------------------------------------------------------------
[   1/288] OK  P16 n=2      s=0 rho=0.5 f=super | it=  4 ||g||=9.93e-13 cg_tot=3 t=0.01s
[   2/288] OK  P16 n=2      s=1 rho=0.5 f=super | it=  6 ||g||=7.77e-16 cg_tot=6 t=0.00s
[   3/288] OK  P16 n=2      s=2 rho=0.5 f=super | it=  4 ||g||=2.78e-13 cg_tot=5 t=0.00s
[   4/288] OK  P16 n=2      s=3 rho=0.5 f=super | it=  5 ||g||=5.97e-15 cg_tot=4 t=0.00s
[   5/288] OK  P16 n=2      s=4 rho=0.5 f=super | it=  6 ||g||=1.02e-15 cg_tot=6 t=0.00s
[   6/288] OK  P16 n=2      s=5 rho=0.5 f=super | it=  3 ||g||=2.50e-10 cg_tot=2 t=0.00s


KeyboardInterrupt: 

In [ ]:
# Truncated Newton: aggregation and ranking

tn_param_cols = ['forcing', 'rho']
tn_summary, tn_detail = aggregate_grid(tn_df, tn_param_cols)

print("=== Truncated Newton: Ranking ===")
print("Sort: avg_success DESC -> avg_iter ASC -> avg_time ASC")
display(tn_summary)

# Best parameters: 3 selections.
best_tn = best_overall(tn_summary)
best_tn_P16 = best_for_problem(tn_df, tn_param_cols, 'P16')
best_tn_P28 = best_for_problem(tn_df, tn_param_cols, 'P28')

print("\n=== Best parameters ===")
print(f"  Overall  : forcing={best_tn['forcing']}, rho={best_tn['rho']}  "
      f"(success={best_tn['avg_success']:.2%}, "
      f"iter={best_tn['avg_iter']:.1f}, time={best_tn['avg_time']:.2f}s)")
print(f"  Best P16 : forcing={best_tn_P16['forcing']}, rho={best_tn_P16['rho']}  "
      f"(success_P16={best_tn_P16['success_P16']:.2%}, "
      f"iter_P16={best_tn_P16['iter_P16']:.1f})")
print(f"  Best P28 : forcing={best_tn_P28['forcing']}, rho={best_tn_P28['rho']}  "
      f"(success_P28={best_tn_P28['success_P28']:.2%}, "
      f"iter_P28={best_tn_P28['iter_P28']:.1f})")

# Pivot tables.
print("\n=== Success rate per (problem, n) x (forcing, rho) ===")
display(pivot_success_by_dim(tn_detail, tn_param_cols))

print("\n=== Mean iterations (success-only) per (problem, n) x (forcing, rho) ===")
display(pivot_iter_by_dim(tn_detail, tn_param_cols))

print("\nNote: NaN in 'mean_iter' = (problem, n) cells where ALL runs failed.")
print("Typical causes: max_iter on P16@n=100000 and time_limit on P28@n>=10000.")


=== Truncated Newton: Ranking ===
Sort: avg_success DESC -> avg_iter ASC -> avg_time ASC


,forcing,rho,success_P16,iter_P16,time_P16,success_P28,iter_P28,time_P28,avg_success,avg_iter,avg_time,avg_grad
rank,,,,,,,,,,,,
0,quadratic,0.8,0.750,48.777778,0.454115,0.500000,21.500000,0.226447,0.625000,37.866667,0.363047,1.084862e-09
1,superlinear,0.8,0.750,63.777778,0.524396,0.458333,21.683333,0.172765,0.604167,47.275862,0.385079,2.626207e-09
2,quadratic,0.5,0.500,16.000000,0.020920,0.500000,21.500000,0.163632,0.500000,17.333333,0.089831,8.589117e-10
3,superlinear,0.5,0.375,18.388889,0.022604,0.458333,21.683333,0.177309,0.416667,16.200000,0.093178,1.384111e-09



=== Best parameters ===
  Overall  : forcing=quadratic, rho=0.8  (success=62.50%, iter=37.9, time=0.36s)
  Best P16 : forcing=quadratic, rho=0.8  (success_P16=75.00%, iter_P16=48.8)
  Best P28 : forcing=quadratic, rho=0.5  (success_P28=50.00%, iter_P28=21.5)

=== Success rate per (problem, n) x (forcing, rho) ===


forcing        quadratic      superlinear          
rho                  0.5  0.8         0.5       0.8
problem n                                          
P16     2            1.0  1.0    1.000000  1.000000
        1000         0.5  1.0    0.333333  1.000000
        10000        0.5  1.0    0.166667  1.000000
        100000       0.0  0.0    0.000000  0.000000
P28     2            1.0  1.0    1.000000  1.000000
        1000         1.0  1.0    0.833333  0.833333
        10000        0.0  0.0    0.000000  0.000000
        100000       0.0  0.0    0.000000  0.000000


=== Mean iterations (success-only) per (problem, n) x (forcing, rho) ===


forcing        quadratic            superlinear            
rho                  0.5        0.8         0.5         0.8
problem n                                                  
P16     2       4.666667   5.000000    4.666667    5.000000
        1000   19.333333  48.666667   22.500000   49.333333
        10000  24.000000  92.666667   28.000000  137.000000
P28     2       5.833333   5.833333    6.166667    6.166667
        1000   37.166667  37.166667   37.200000   37.200000


Note: NaN in 'mean_iter' = (problem, n) cells where ALL runs failed.
Typical causes: max_iter on P16@n=100000 and time_limit on P28@n>=10000.


In [ ]:
# Convergence rate for best configs (n=2 with history)

rate_rows = []
for method_name, method_fn, params in [
    ('ModNewton', modified_newton,
     dict(beta=best_mn['beta'], rho=best_mn['rho'], **MN_FIXED)),
    ('TruncNewton', truncated_newton,
     dict(forcing=best_tn['forcing'], rho=best_tn['rho'], **TN_FIXED)),
]:
    for prob_id in ('P16', 'P28'):
        pinfo = PROBLEMS[prob_id]
        x0 = starts_cache[(prob_id, 2)][0]
        stop = GradNormAbsolute(tol=1e-8)
        res = method_fn(pinfo['f'], x0, stop,
                        grad_f=pinfo['grad'], hess_f=pinfo['hess'],
                        alpha0=1.0, max_iter=MAX_ITER,
                        return_history=True, **params)
        g_norms = [h['grad_norm'] for h in res.get('history', [])]
        rate = experimental_rate(g_norms)
        rate_rows.append(dict(method=method_name, problem=prob_id,
                              n_iter=res['n_iter'], rate=rate))

print("=== Convergence Rate Estimate (n=2, x_bar) ===")
print(pd.DataFrame(rate_rows).to_string(index=False))

=== Convergence Rate Estimate (n=2, x_bar) ===
     method problem  n_iter     rate
  ModNewton     P16       9 3.182244
  ModNewton     P28       7 1.683462
TruncNewton     P16       4 3.235160
TruncNewton     P28       7 1.683462


<a id="phase1-results"></a>
## 5. Phase 1: Results & Visualization

In [ ]:
# Best parameters summary

summary_rows = [
    {
        'Method': 'Modified Newton',
        'rho': best_mn['rho'],
        'Method-specific': f"beta={best_mn['beta']:.0e}",
        'Fixed': f"c1=1e-4, max_tau=100, max_bt=50",
        'Success Rate': f"{best_mn['avg_success']:.2%}",
        'Avg Iterations': f"{best_mn['avg_iter']:.1f}",
    },
    {
        'Method': 'Truncated Newton',
        'rho': best_tn['rho'],
        'Method-specific': f"forcing={best_tn['forcing']}",
        'Fixed': f"c1=1e-4, cg_max=n, max_bt=50",
        'Success Rate': f"{best_tn['avg_success']:.2%}",
        'Avg Iterations': f"{best_tn['avg_iter']:.1f}",
    },
]

display(Markdown("### Selected Parameters"))
display(pd.DataFrame(summary_rows).set_index('Method'))

### Selected Parameters

,rho,Method-specific,Fixed,Success Rate,Avg Iterations
Method,,,,,
Modified Newton,0.8,beta=1e-06,"c1=1e-4, max_tau=100, max_bt=50",52.38%,16.8
Truncated Newton,0.8,forcing=quadratic,"c1=1e-4, cg_max=n, max_bt=50",62.50%,37.9


<a id="phase2"></a>
## 6. Phase 2: Stopping Criteria Analysis (Post-hoc)

The stopping criterion does **not** affect the convergence trajectory — it only decides when to declare convergence.

**Efficient approach**: run each algorithm **once** with `return_history=True` and a tight tolerance, then evaluate all stopping criteria **post-hoc** on the recorded trajectory. This eliminates the 22× re-run overhead.

### Tolerance Bands (from course slides)

| Band | TOL | Quality |
|------|-----|---------|
| **Rough** | $10^{-4}$ | Rough precision |
| **Good** | $10^{-8}$ | Good solution |
| **Very good** | $10^{-12}$ | Very demanding, often unnecessary |

### Criteria evaluated

| # | Criterion | Band | Purpose |
|---|-----------|------|---------|
| 1 | `grad_abs` | rough / good / very_good | Gold standard |
| 2 | `x_abs` | rough / good | Secondary confirmation |
| 3 | `grad_rel` | rough | Show it works only at loose tolerance |
| 4 | `combined_abs` | good | Combined (OR) approach |

### How to read the results

For each (method, problem, n, start), the algorithm history records `grad_norm`, `f`, `x`, `x_prev` at each iteration. We scan the history and find **the first iteration where each criterion would fire**. This gives:

- **`stop_iter`**: iteration at which the criterion would stop
- **`grad_norm_at_stop`**: quality of the solution at that point
- **`success`**: whether the criterion fired before `max_iter`

A **good** stopping criterion:
1. **Fires** (success = True) — it actually detects convergence
2. Fires at a point with **small `grad_norm_at_stop`** — the solution is accurate
3. Fires **early** (low `stop_iter`) — it does not waste iterations beyond convergence

A **bad** criterion (e.g. `grad_rel` at tight tolerance on P28) either:
- Fires too early with **huge** `grad_norm_at_stop` (false convergence)
- Never fires (the relative threshold is met trivially or never)

In [ ]:
# Tolerance bands adapted per criterion type

TOLERANCE_BANDS = {
    #              (tol_grad,  tol_f,    tol_x)
    'rough':       (1e-4,      1e-8,     1e-4),
    'good':        (1e-8,      1e-16,    1e-8),
    'very_good':   (1e-12,     None,     1e-12),  # None = f-change not feasible
}

def make_stopping_config(crit_type, band):
    """Create StoppingCriterion for a given type and band.

    Returns None if the combination is not feasible.
    """
    tol_g, tol_f, tol_x = TOLERANCE_BANDS[band]
    factories = {
        'grad_abs':     (tol_g, lambda t: GradNormAbsolute(t)),
        'grad_rel':     (tol_g, lambda t: GradNormRelative(t)),
        'x_abs':        (tol_x, lambda t: XChangeAbsolute(t)),
        'combined_abs': (None,  lambda _: CombinedStoppingCriterion([
                            GradNormAbsolute(tol_g),
                            *([FChangeAbsolute(tol_f)] if tol_f is not None else []),
                            XChangeAbsolute(tol_x)])),
    }
    if crit_type not in factories:
        return None
    tol, factory = factories[crit_type]
    return factory(tol)

# Reduced set: 7 (crit_type, band) pairs (from 22)
SC_CONFIGS = (
    [(c, b) for c in ['grad_abs', 'grad_rel', 'f_abs', 'f_rel',
                       'x_abs', 'x_rel']
            for b in ['rough', 'good', 'very_good']]
    + [(c, b) for c in ['combined_abs', 'combined_rel']
              for b in ['rough', 'good']]
)

print(f"{len(SC_CONFIGS)} stopping criterion configurations:")
for ct, b in SC_CONFIGS:
    tol_g, tol_f, tol_x = TOLERANCE_BANDS[b]
    print(f"  {ct:15s} @ {b:10s}  "
          f"(tol_g={tol_g:.0e}, tol_x={tol_x:.0e})")

22 stopping criterion configurations:
  grad_abs        @ rough       (tol_g=1e-04, tol_x=1e-04)
  grad_abs        @ good        (tol_g=1e-08, tol_x=1e-08)
  grad_abs        @ very_good   (tol_g=1e-12, tol_x=1e-12)
  grad_rel        @ rough       (tol_g=1e-04, tol_x=1e-04)
  grad_rel        @ good        (tol_g=1e-08, tol_x=1e-08)
  grad_rel        @ very_good   (tol_g=1e-12, tol_x=1e-12)
  f_abs           @ rough       (tol_g=1e-04, tol_x=1e-04)
  f_abs           @ good        (tol_g=1e-08, tol_x=1e-08)
  f_abs           @ very_good   (tol_g=1e-12, tol_x=1e-12)
  f_rel           @ rough       (tol_g=1e-04, tol_x=1e-04)
  f_rel           @ good        (tol_g=1e-08, tol_x=1e-08)
  f_rel           @ very_good   (tol_g=1e-12, tol_x=1e-12)
  x_abs           @ rough       (tol_g=1e-04, tol_x=1e-04)
  x_abs           @ good        (tol_g=1e-08, tol_x=1e-08)
  x_abs           @ very_good   (tol_g=1e-12, tol_x=1e-12)
  x_rel           @ rough       (tol_g=1e-04, tol_x=1e-04)
  x_rel           

In [ ]:
# Phase 2: post-hoc stopping criteria evaluation
#
# Strategy: run each (method, problem, n, start) ONCE with a wrapper
# that logs lightweight metrics (grad_norm, f, ||dx||) at each iteration.
# Then evaluate all SC_CONFIGS on the recorded log — zero extra runs.
#
# This avoids storing full x vectors (which would be 800 MB/run at n=100k).

class MetricsLogger(StoppingCriterion):
    """Wraps a tight stopping criterion and logs per-iteration metrics."""

    def __init__(self, inner_stop):
        self.inner = inner_stop
        self.tol = inner_stop.tol
        self.log = []

    @property
    def name(self):
        return self.inner.name

    def initialize(self, x0, F0, g0):
        self.inner.initialize(x0, F0, g0)
        g0_norm = float(np.linalg.norm(g0))
        self.log = [{'k': 0, 'grad_norm': g0_norm, 'f': float(F0),
                     'x_change': 0.0, 'x_norm': float(np.linalg.norm(x0))}]

    def should_stop(self, k, x, F, g, x_prev, F_prev):
        self.log.append({
            'k': k,
            'grad_norm': float(np.linalg.norm(g)),
            'f': float(F),
            'x_change': float(np.linalg.norm(x - x_prev)) if k > 0 else 0.0,
            'x_norm': float(np.linalg.norm(x)),
        })
        return self.inner.should_stop(k, x, F, g, x_prev, F_prev)


def eval_criterion_on_log(log, crit_type, band):
    """Evaluate a stopping criterion on a lightweight metrics log.

    Returns (stop_iter, grad_norm_at_stop, f_at_stop, reason) or None.

    A 'log' entry has keys: k, grad_norm, f, x_k, x_prev (optional).
    """
    tol_g, tol_f, tol_x = TOLERANCE_BANDS[band]
    g0_norm = log[0]['grad_norm'] if log else 1.0
    f0 = log[0]['f'] if log else 0.0

    # Skip non-feasible combinations early.
    if crit_type in ('f_abs', 'f_rel') and tol_f is None:
        return None
    if crit_type == 'combined_abs' and tol_g is None:
        return None
    if crit_type == 'combined_rel' and tol_g is None:
        return None

    def grad_abs(entry):
        return entry['grad_norm'] <= tol_g
    def grad_rel(entry):
        return entry['grad_norm'] <= tol_g * max(g0_norm, 1.0)
    def f_abs(entry, prev_f):
        if prev_f is None: return False
        return abs(entry['f'] - prev_f) <= tol_f
    def f_rel(entry, prev_f):
        if prev_f is None: return False
        return abs(entry['f'] - prev_f) <= tol_f * max(abs(entry['f']), 1.0)
    def x_abs(entry):
        sn = entry.get('x_change')
        return sn is not None and sn <= tol_x
    def x_rel(entry):
        sn = entry.get('x_change')
        xn = entry.get('x_norm')
        if sn is None: return False
        return sn <= tol_x * max(xn if xn is not None else 1.0, 1.0)

    prev_f = log[0]['f'] if log else None
    for entry in log[1:]:
        fired = False
        reason = None
        if crit_type == 'grad_abs' and grad_abs(entry):
            fired, reason = True, 'grad_abs'
        elif crit_type == 'grad_rel' and grad_rel(entry):
            fired, reason = True, 'grad_rel'
        elif crit_type == 'f_abs' and f_abs(entry, prev_f):
            fired, reason = True, 'f_abs'
        elif crit_type == 'f_rel' and f_rel(entry, prev_f):
            fired, reason = True, 'f_rel'
        elif crit_type == 'x_abs' and x_abs(entry):
            fired, reason = True, 'x_abs'
        elif crit_type == 'x_rel' and x_rel(entry):
            fired, reason = True, 'x_rel'
        elif crit_type == 'combined_abs':
            if grad_abs(entry):
                fired, reason = True, 'grad_abs'
            elif tol_f is not None and f_abs(entry, prev_f):
                fired, reason = True, 'f_abs'
            elif x_abs(entry):
                fired, reason = True, 'x_abs'
        elif crit_type == 'combined_rel':
            if grad_rel(entry):
                fired, reason = True, 'grad_rel'
            elif tol_f is not None and f_rel(entry, prev_f):
                fired, reason = True, 'f_rel'
            elif x_rel(entry):
                fired, reason = True, 'x_rel'
        if fired:
            return entry['k'], entry['grad_norm'], entry['f'], reason
        prev_f = entry['f']
    return None


# Step 1: run each cell once with MetricsLogger wrapping GradNormAbsolute(1e-12)

best_mn_params = dict(beta=best_mn['beta'], rho=best_mn['rho'], **MN_FIXED)
best_tn_params = dict(forcing=best_tn['forcing'], rho=best_tn['rho'], **TN_FIXED)

methods_config = [
    ('ModNewton',  modified_newton,  best_mn_params),
    ('TruncNewton', truncated_newton, best_tn_params),
]

metrics_logs = {}
idx = 0
total_runs = sum(
    len(starts_cache[(pid, n)])
    for mn, _, _ in methods_config
    for pid in ('P16', 'P28')
    for n in DIMENSIONS
    if not should_skip(pid, n,
        'modified_newton' if 'Mod' in mn else 'truncated_newton')
)
t0_total = time.perf_counter()
print(f"Phase 2: collecting metrics — {total_runs} runs")
print("-" * 80)

for method_name, method_fn, params in methods_config:
    m_key = 'modified_newton' if 'Mod' in method_name else 'truncated_newton'
    for prob_id in ('P16', 'P28'):
        pinfo = PROBLEMS[prob_id]
        for n in DIMENSIONS:
            if should_skip(prob_id, n, m_key):
                continue
            for si, x0 in enumerate(starts_cache[(prob_id, n)]):
                idx += 1
                logger = MetricsLogger(GradNormAbsolute(tol=1e-12))
                t0 = time.perf_counter()
                try:
                    res = method_fn(
                        pinfo['f'], x0, logger,
                        grad_f=pinfo['grad'],
                        hess_f=(pinfo['hess']
                                if m_key == 'modified_newton'
                                or expected_hessian_mb(n) <= OOM_THRESHOLD_MB
                                else None),
                        alpha0=1.0, max_iter=MAX_ITER,
                        return_history=False, time_limit=TIME_LIMIT,
                        **params)
                    elapsed = time.perf_counter() - t0
                    tag = 'OK' if res['success'] else 'FAIL'
                    print(f"[{idx:>4d}/{total_runs}] {tag}  {method_name:11s} "
                          f"{prob_id} n={n:<6d} s={si} | "
                          f"it={res['n_iter']:>3d} ||g||={res['grad_norm']:.2e} "
                          f"t={elapsed:.2f}s")
                    metrics_logs[(method_name, prob_id, n, si)] = logger.log
                except Exception as e:
                    print(f"[{idx:>4d}/{total_runs}] ERR  {method_name:11s} "
                          f"{prob_id} n={n:<6d} s={si} | {e}")
                    metrics_logs[(method_name, prob_id, n, si)] = []

print("-" * 80)
print(f"Metrics collected in {time.perf_counter() - t0_total:.1f}s")

# Step 2: evaluate all criteria post-hoc on every metrics log
sc_rows = []
for (method_name, prob_id, n, si), log in metrics_logs.items():
    for crit_type, band in SC_CONFIGS:
        result = eval_criterion_on_log(log, crit_type, band)
        if result is not None:
            stop_iter, grad_at_stop, f_at_stop, reason = result
            sc_rows.append(dict(
                method=method_name, problem=prob_id, n=n,
                start_idx=si, crit_type=crit_type, band=band,
                stop_iter=stop_iter, success=True,
                grad_norm_at_stop=grad_at_stop,
                f_at_stop=f_at_stop,
                stop_reason=reason))
        else:
            # FIX: do NOT record the baseline's final grad/f when the
            # post-hoc criterion never fires. Otherwise the mean grad
            # is contaminated by the baseline's runaway final value
            # (~1e27 on P28 because tol=1e-12 is never reached).
            sc_rows.append(dict(
                method=method_name, problem=prob_id, n=n,
                start_idx=si, crit_type=crit_type, band=band,
                stop_iter=len(log) - 1, success=False,
                grad_norm_at_stop=np.nan,
                f_at_stop=np.nan,
                stop_reason='max_iter'))

sc_df = pd.DataFrame(sc_rows)
print(f"\nPost-hoc evaluation: {len(sc_df)} entries "
      f"({len(metrics_logs)} runs x {len(SC_CONFIGS)} criteria)")

Phase 2: collecting metrics — 90 runs
--------------------------------------------------------------------------------
[   1/90] OK  ModNewton   P16 n=2      s=0 | it= 10 ||g||=1.43e-13 t=0.00s
[   2/90] FAIL  ModNewton   P16 n=2      s=1 | it=1000 ||g||=2.55e-12 t=0.32s
[   3/90] OK  ModNewton   P16 n=2      s=2 | it=  6 ||g||=2.30e-14 t=0.00s
[   4/90] FAIL  ModNewton   P16 n=2      s=3 | it=1000 ||g||=2.62e-12 t=0.32s
[   5/90] OK  ModNewton   P16 n=2      s=4 | it=  5 ||g||=1.57e-13 t=0.00s
[   6/90] FAIL  ModNewton   P16 n=2      s=5 | it=1000 ||g||=1.71e-11 t=0.45s
[   7/90] FAIL  ModNewton   P16 n=1000   s=0 | it=1000 ||g||=9.64e-11 t=0.42s
[   8/90] FAIL  ModNewton   P16 n=1000   s=1 | it=1000 ||g||=7.22e-09 t=0.49s
[   9/90] FAIL  ModNewton   P16 n=1000   s=2 | it=1000 ||g||=2.18e-08 t=0.51s
[  10/90] FAIL  ModNewton   P16 n=1000   s=3 | it=1000 ||g||=1.38e-07 t=0.54s
[  11/90] FAIL  ModNewton   P16 n=1000   s=4 | it=1000 ||g||=3.96e-08 t=0.50s
[  12/90] FAIL  ModNewton   P16 

In [ ]:
# Stopping criteria: aggregation and tables
#
# fire_rate            -> frac of runs where the post-hoc criterion triggers
# mean_stop_iter       -> avg iter at trigger (on fired runs ONLY)
# mean_grad_at_stop    -> ||grad|| at trigger (on fired runs ONLY; NaN excluded)
# mean_f_at_stop       -> f-value at trigger (on fired runs ONLY)
#
# A useful criterion has fire_rate close to 1, small mean_stop_iter,
# and small mean_grad_at_stop.

ok_sc = sc_df[sc_df['success']]

# Layer 1: aggregate per method only (overall picture).
sc_agg = sc_df.groupby(['method', 'crit_type', 'band']).agg(
    fire_rate=('success', 'mean'),
).reset_index()
qual = ok_sc.groupby(['method', 'crit_type', 'band']).agg(
    mean_stop_iter=('stop_iter', 'mean'),
    std_stop_iter=('stop_iter', 'std'),
    mean_grad_at_stop=('grad_norm_at_stop', 'mean'),
    mean_f_at_stop=('f_at_stop', 'mean'),
).reset_index()
sc_agg = sc_agg.merge(qual, on=['method', 'crit_type', 'band'], how='left')

# Layer 2: split by problem.
sc_agg_prob = sc_df.groupby(['method', 'problem', 'crit_type', 'band']).agg(
    fire_rate=('success', 'mean'),
).reset_index()
qual_prob = ok_sc.groupby(['method', 'problem', 'crit_type', 'band']).agg(
    mean_stop_iter=('stop_iter', 'mean'),
    mean_grad_at_stop=('grad_norm_at_stop', 'mean'),
).reset_index()
sc_agg_prob = sc_agg_prob.merge(
    qual_prob, on=['method', 'problem', 'crit_type', 'band'], how='left')

for method in ('ModNewton', 'TruncNewton'):
    print(f"\n=== Stopping Criteria — {method} (overall) ===")
    sub = sc_agg[sc_agg['method'] == method].sort_values(['crit_type', 'band'])
    print(sub.to_string(index=False))

for method in ('ModNewton', 'TruncNewton'):
    print(f"\n=== Stopping Criteria — {method} (per problem) ===")
    sub = sc_agg_prob[sc_agg_prob['method'] == method].sort_values(
        ['crit_type', 'band', 'problem'])
    print(sub.to_string(index=False))

print("\n--- Interpretation guide ---")
print("A good stopping criterion has:")
print("  1. fire_rate close to 1.0  (it actually fires)")
print("  2. small mean_grad_at_stop  (accurate solution when it fires)")
print("  3. low mean_stop_iter  (does not waste iterations past convergence)")
print()
print("Failure modes to watch for:")
print("  - fire_rate = 0     : criterion never triggers (too tight, e.g.")
print("                         grad_abs @ very_good on P28).")
print("  - mean_grad_at_stop : huge value -> false positive (grad_rel @ good")
print("                         on P28 fires when ||g||~1e12 due to ||g0||=O(n^7)).")
print("  - mean_stop_iter    : at max_iter -> criterion doesn't discriminate.")



=== Stopping Criteria — ModNewton (overall) ===
   method    crit_type      band  fire_rate  mean_stop_iter  std_stop_iter  mean_grad_at_stop  mean_f_at_stop
ModNewton combined_abs      good   0.857143       54.861111      60.148936       4.958545e-04   -7.672095e+03
ModNewton combined_abs     rough   0.857143       52.944444      60.282641       1.057577e+05   -7.668168e+03
ModNewton combined_rel      good   0.857143       49.861111      59.808816       7.682062e+11    5.281138e+09
ModNewton combined_rel     rough   0.857143       32.055556      37.943525       1.293206e+16    2.278492e+15
ModNewton        f_abs      good   0.785714       59.818182      60.631806       5.409321e-04   -8.369468e+03
ModNewton        f_abs     rough   0.857143       54.888889      59.745545       4.958544e-04   -7.672095e+03
ModNewton        f_abs very_good   0.000000             NaN            NaN                NaN             NaN
ModNewton        f_rel      good   0.785714       59.818182      60.631

<a id="export"></a>
## 7. Export & Summary

In [ ]:
# Save results to CSV

results_dir = ROOT / 'results'
results_dir.mkdir(exist_ok=True)

mn_df.to_csv(results_dir / 'fine_tuning_modified_newton.csv', index=False)
tn_df.to_csv(results_dir / 'fine_tuning_truncated_newton.csv', index=False)
sc_df.to_csv(results_dir / 'fine_tuning_stopping_criteria.csv', index=False)

print(f"Saved to {results_dir}:")
print(f"  fine_tuning_modified_newton.csv   ({len(mn_df)} rows)")
print(f"  fine_tuning_truncated_newton.csv  ({len(tn_df)} rows)")
print(f"  fine_tuning_stopping_criteria.csv ({len(sc_df)} rows)")

Saved to c:\Users\andre\Documents\University\2nd Year\NO4LSP\project\Unconstrained_Optimization_Project\results:
  fine_tuning_modified_newton.csv   (168 rows)
  fine_tuning_truncated_newton.csv  (192 rows)
  fine_tuning_stopping_criteria.csv (1980 rows)


In [ ]:
# Final summary

print("=" * 70)
print("FINE-TUNING SUMMARY")
print("=" * 70)
print(f"Dimensions: {DIMENSIONS}")
print(f"Starting points: {1 + NUM_RANDOM} per (problem, n)")
print(f"MAX_ITER: {MAX_ITER}")
print()
print("BEST MODIFIED NEWTON:")
print(f"  beta={best_mn['beta']:.0e}, rho={best_mn['rho']}")
print(f"  fixed: c1=1e-4, max_tau_iter=100, max_iter_backtrack=50")
print(f"  success rate: {best_mn['avg_success']:.2%}")
print(f"  avg iterations: {best_mn['avg_iter']:.1f}")
print()
print("BEST TRUNCATED NEWTON:")
print(f"  forcing={best_tn['forcing']}, rho={best_tn['rho']}")
print(f"  fixed: c1=1e-4, cg_max_iter=n, max_iter_backtrack=50")
print(f"  success rate: {best_tn['avg_success']:.2%}")
print(f"  avg iterations: {best_tn['avg_iter']:.1f}")
print()
print("=" * 70)

FINE-TUNING SUMMARY
Dimensions: [2, 1000, 10000, 100000]
Starting points: 6 per (problem, n)
MAX_ITER: 1000

BEST MODIFIED NEWTON:
  beta=1e-06, rho=0.8
  fixed: c1=1e-4, max_tau_iter=100, max_iter_backtrack=50
  success rate: 52.38%
  avg iterations: 16.8

BEST TRUNCATED NEWTON:
  forcing=quadratic, rho=0.8
  fixed: c1=1e-4, cg_max_iter=n, max_iter_backtrack=50
  success rate: 62.50%
  avg iterations: 37.9

